In [1]:
import pandas as pd
import numpy as np

df_llm = pd.read_csv("production_run.csv")
df_human_raw = pd.read_excel("june 23 human classification.xlsx")

# explicit mapping from human Subprocess codes to the LLM label scheme
# fill in actual subprocess names per your Doxey-based register
subprocess_map = {
    0: "OUT_OF_SCOPE",
    1: "Master Data Management",
    2: "Record",
    3: "Close & Consolidation",
    4: "Report",
    5: "Analysis & Business Insight",
    6: "Planning & Budgeting",
    7: "AMBIGUOUS",
}

df_human = pd.DataFrame()
df_human['task_id'] = pd.to_numeric(df_human_raw['Task ID'], errors='coerce')
print("Unparseable Task IDs:", df_human['task_id'].isna().sum())  # check before casting
df_human['task_id'] = df_human['task_id'].astype(int)

df_human['task_statement'] = df_human_raw['Task']
df_human['classification'] = df_human_raw['Subprocess'].map(subprocess_map)
print("Unmapped subprocess codes:", df_human['classification'].isna().sum())  # check the map worked
df_human['reasoning'] = df_human_raw['Brief rationale']

df_human['tier'] = 'human_expert'
df_human['provider'] = 'human'
df_human['model'] = 'human_domain_expert'
df_human['temperature'] = np.nan
df_human['run_id'] = 1
df_human['input_tokens'] = np.nan
df_human['output_tokens'] = np.nan

# sanity check before merging
human_ids = set(df_human['task_id'])
llm_ids = set(df_llm['task_id'])
print("In human but not LLM:", human_ids - llm_ids)
print("In LLM but not human:", llm_ids - human_ids)

df_long_unified = pd.concat([df_llm, df_human], ignore_index=True)
df_long_unified.to_csv("june 23 unified_rater_long_format.csv", index=False)

Unparseable Task IDs: 0
Unmapped subprocess codes: 0
In human but not LLM: set()
In LLM but not human: set()


In [ ]:
!pip install krippendorff

In [3]:
"""
Agreement Analysis for Phase 1 R2R Task Classification — PRODUCTION RUN
=========================================================================
Reads a single unified long-format file — the same shape as the calibration
stage's unified_rater_long_format.csv — where human rows are already merged
in alongside the 3 mid-tier model raters:

    task_id, task_statement, tier, provider, model, temperature, run_id,
    classification, reasoning, input_tokens, output_tokens

Rater identity:
    - Model rows : provider in {claude, openai, gemini}, tier="mid",
                    temperature=0.2, run_id=1 (one call per model per task)
    - Human rows : provider="human", tier="human_expert" (or any non-mid
                    value), model="human_domain_expert", temperature/run_id
                    may be blank — both are ignored for human rows

No separate --human file is needed any more; human rows are auto-detected
by model == "human_domain_expert", exactly as in the calibration script.

Tests carried over from the calibration script, re-scoped to what the
production grid can support (1 run x 1 temperature x 1 tier per model, so
run-to-run / by-temperature / by-tier comparisons aren't possible — see
"Dropped" below):

    Test 3b  Model x model pairwise agreement matrix (3 model raters)
    Test 6   Full cross-model agreement, all 3 model raters pooled
             (K-alpha nominal + 95% bootstrap CI, mean pairwise %,
             full-agreement rate)
    Test 7   Agreement of each model vs. the human rater
             (raw % agreement + K-alpha nominal, per model)
    Test 8   Per-subprocess disagreement breakdown (consensus_strength,
             mode-share 0->1), computed across ALL 4 raters (human + 3
             models)
    Test 8b  Subprocess confusion / transition matrix (all 4 raters)
    Test 9   Punt rate per rater (AMBIGUOUS + OUT_OF_SCOPE usage)
    Test 10  Task-volume distribution per subprocess per rater, plus a
             Pearson correlation matrix across raters' subprocess volume
             counts — the "Pearson correlation of total task volumes per
             subprocess" check from the workflow doc — including each
             model's Pearson r against the human volume profile.

Dropped vs. the calibration script (structurally impossible at N=1):
    Test 1   Intra-model run-to-run agreement (only 1 run per model)
    Test 2   Agreement by temperature (only one temperature, 0.2)
    Test 3   Agreement by tier (only mid tier was run for the models)
    Test 4   Agreement by provider — at 1 model per provider this is
             identical to Test 3b's per-model rows, so it isn't a
             separate groupby here.
    Test 5   Agreement by model across runs/temps — collapses to Test 6
             at N=1 run/temp per model.

A reconciliation-flag summary is included under Test 8, mirroring your
consensus script's own routing rule — but note this is a DIAGNOSTIC only.
It is NOT a dependency of consensus_production.md: that script reads
production_initial.csv directly and computes its own 3+/4-vs-split tally
from all 4 raters. The numbers here are for sanity-checking / reporting
(e.g. "what fraction of tasks needed reconciliation, and where") rather
than for driving the consensus round itself.

Usage:
    python production_agreement_analysis.py --input unified_rater_long_format.csv
"""

import argparse
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from krippendorff import alpha as krippendorff_alpha

warnings.filterwarnings("ignore")

PUNT_LABELS = {"AMBIGUOUS", "OUT_OF_SCOPE"}
HUMAN_MODEL = "human_domain_expert"

# Canonical display order. NOTE: the production classifier script uses the
# label "Record" while the consensus script's ALLOWED_LABELS uses
# "Recording" — pick whichever your actual data uses and edit this list to
# match (or just leave both in; build_label_map() below treats this purely
# as a preferred *ordering* and will auto-add any label it finds in the
# data that isn't already listed, so a mismatch here will not crash the
# script — it'll just put the unrecognised spelling at the end and print a
# warning so you catch the inconsistency).
SUBPROCESS_ORDER = [
    "Master Data Management",
    "Record",
    "Close & Consolidation",
    "Report",
    "Analysis & Business Insight",
    "Planning & Budgeting",
    "AMBIGUOUS",
    "OUT_OF_SCOPE",
]


def build_label_map(df: pd.DataFrame) -> tuple[dict, list]:
    """
    Derive the label -> int code map from the data itself rather than
    hardcoding it, so a "Record" vs "Recording" (or any other) spelling
    mismatch between your classifier and consensus scripts doesn't
    silently corrupt the K-alpha encoding (unmapped labels -> NaN, which
    would quietly drop those tasks from every reliability stat).

    Returns (label_map, ordered_labels). ordered_labels follows
    SUBPROCESS_ORDER for any label present in the data, then appends any
    observed-but-unlisted labels (printing a warning).
    """
    observed = set(df["classification"].dropna().unique()) - {"API_ERROR", "PARSE_ERROR"}
    ordered = [l for l in SUBPROCESS_ORDER if l in observed]
    unexpected = sorted(observed - set(SUBPROCESS_ORDER))
    if unexpected:
        print(f"  \u26a0  Labels found in data but not in SUBPROCESS_ORDER: {unexpected}\n"
              f"     (added to the end of the ordering — check for a spelling mismatch,\n"
              f"     e.g. 'Record' vs 'Recording', between your classifier and consensus scripts)")
        ordered += unexpected
    label_map = {label: i for i, label in enumerate(ordered)}
    return label_map, ordered


# ── Helpers ──────────────────────────────────────────────────────────────────
def encode_labels(series: pd.Series, label_map: dict) -> pd.Series:
    """Map text labels -> int codes for Krippendorff; unknown -> NaN."""
    return series.map(label_map)


# Set once per run by run_analysis() via build_label_map(df), then used by
# pivot_numeric()/summarise_agreement() below. Module-level rather than
# threaded through every helper's signature, since it's fixed for the
# duration of one script invocation.
_ACTIVE_LABEL_MAP: dict = {}


def pairwise_agreement(labels_a: pd.Series, labels_b: pd.Series) -> float:
    """Simple percent agreement between two aligned series."""
    mask = labels_a.notna() & labels_b.notna()
    if mask.sum() == 0:
        return np.nan
    return (labels_a[mask] == labels_b[mask]).mean()


def krippendorff_nominal_ci(
    reliability_data: np.ndarray, n_bootstrap: int = 1000, ci: float = 0.95
) -> tuple[float, float, float]:
    """Krippendorff's alpha at NOMINAL level with bootstrap CI.

    reliability_data : shape (n_raters, n_tasks), NaN for missing.
    """
    try:
        alpha_hat = krippendorff_alpha(
            reliability_data, level_of_measurement="nominal"
        )
    except Exception:
        return np.nan, np.nan, np.nan

    n_tasks = reliability_data.shape[1]
    boot_alphas = []
    rng = np.random.default_rng(seed=42)
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n_tasks, size=n_tasks)
        try:
            a = krippendorff_alpha(
                reliability_data[:, idx], level_of_measurement="nominal"
            )
            boot_alphas.append(a)
        except Exception:
            pass

    if not boot_alphas:
        return alpha_hat, np.nan, np.nan

    tail = (1 - ci) / 2
    ci_lower = float(np.nanpercentile(boot_alphas, tail * 100))
    ci_upper = float(np.nanpercentile(boot_alphas, (1 - tail) * 100))
    return alpha_hat, ci_lower, ci_upper


def _build_rater_id(df: pd.DataFrame) -> pd.Series:
    """
    Compose a unique rater ID.

    Production grid is 1 run x 1 temperature per model, so 'model' alone
    is already a unique rater — but we still suffix temperature/run_id
    where present so the script keeps working unchanged if a future
    production re-run adds a second pass (e.g. a robustness re-run at a
    different temperature) without silently merging distinct raters.
    """
    is_human = df["model"] == HUMAN_MODEL
    llm_id = (
        df["model"].astype(str)
        + "__"
        + df.get("temperature", pd.Series("", index=df.index)).astype(str)
        + "__"
        + df.get("run_id", pd.Series("", index=df.index)).astype(str)
    )
    return llm_id.where(~is_human, other=HUMAN_MODEL)


def pivot_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """task_id x rater pivot with numeric codes — for Krippendorff."""
    tmp = df.copy()
    tmp["_rater"] = _build_rater_id(tmp)
    tmp["_num"] = encode_labels(tmp["classification"], _ACTIVE_LABEL_MAP)
    return tmp.pivot_table(
        index="task_id", columns="_rater", values="_num", aggfunc="first"
    )


def pivot_text(df: pd.DataFrame) -> pd.DataFrame:
    """task_id x rater pivot with text labels — for pairwise agreement."""
    tmp = df.copy()
    tmp["_rater"] = _build_rater_id(tmp)
    return tmp.pivot_table(
        index="task_id", columns="_rater", values="classification", aggfunc="first"
    )


def summarise_agreement(df_subset: pd.DataFrame, label: str) -> dict:
    """K-alpha (nominal, 95% CI) + mean pairwise % agreement + full-agreement
    rate for a long-format subset."""
    p_num = pivot_numeric(df_subset)
    alpha_hat, ci_lo, ci_hi = krippendorff_nominal_ci(p_num.values.T)

    p_txt = pivot_text(df_subset)
    raters = list(p_txt.columns)
    pw_scores = [
        pairwise_agreement(p_txt[a], p_txt[b])
        for a, b in itertools.combinations(raters, 2)
    ]
    mean_pw = np.nanmean(pw_scores) if pw_scores else np.nan
    full_agree = (p_txt.nunique(axis=1) == 1).mean()

    return {
        "group": label,
        "n_raters": len(raters),
        "n_tasks": len(p_txt),
        "krippendorff_alpha_nominal": round(alpha_hat, 4) if not np.isnan(alpha_hat) else np.nan,
        "alpha_95ci_lower": round(ci_lo, 4) if not np.isnan(ci_lo) else np.nan,
        "alpha_95ci_upper": round(ci_hi, 4) if not np.isnan(ci_hi) else np.nan,
        "mean_pairwise_agreement": round(mean_pw, 4) if not np.isnan(mean_pw) else np.nan,
        "full_agreement_rate": round(full_agree, 4),
    }


def model_mode_series(df: pd.DataFrame, model_name: str) -> pd.Series:
    """task_id-indexed Series of the modal label for one model (N=1 run/temp
    in production, so this is just that model's single label per task, but
    .mode() keeps the function safe if duplicate rows ever exist)."""
    sub = df[df["model"] == model_name]
    return sub.groupby("task_id")["classification"].agg(lambda x: x.mode().iloc[0])


def consensus_strength(row: pd.Series) -> float:
    """Mode-share per task: count(modal label) / count(non-NaN raters)."""
    vals = row.dropna()
    if len(vals) == 0:
        return np.nan
    return vals.value_counts().iloc[0] / len(vals)


# ── Main analysis ─────────────────────────────────────────────────────────────
def run_analysis(df: pd.DataFrame):
    """
    df : production long-format DataFrame (task_id, model, provider, tier,
         temperature, run_id, classification, reasoning, ...). May
         optionally contain human/reference rows with model == HUMAN_MODEL
         (e.g. the validated R2R Task Register from Phase 1 reconciliation),
         which enables Test 7.
    """
    global _ACTIVE_LABEL_MAP
    _ACTIVE_LABEL_MAP, active_order = build_label_map(df)
    global SUBPROCESS_ORDER
    SUBPROCESS_ORDER = active_order

    human_mask = df["model"] == HUMAN_MODEL
    df_llm = df[~human_mask].copy()
    df_all = df.copy()

    _human = (
        df[human_mask][["task_id", "classification"]]
        .drop_duplicates("task_id")
        if human_mask.any() else None
    )

    if _human is not None:
        print(f"  \u2713  Human/reference annotations found: {len(_human):,} tasks")
    else:
        print("  \u26a0  No human rows found (model == 'human_domain_expert'). "
              "Test 7 and the vs-human part of Test 10 will be skipped.")

    n_models = df_llm["model"].nunique()
    if n_models != 3:
        print(f"  \u26a0  Expected 3 mid-tier raters in production, found {n_models}: "
              f"{sorted(df_llm['model'].unique())}")

    # ── 0. Label sanity check (also see build_label_map warning above) ─────
    api_errors = (df["classification"] == "API_ERROR").sum()
    parse_errors = (df["classification"] == "PARSE_ERROR").sum()
    if api_errors or parse_errors:
        print(f"  \u26a0  {api_errors} API_ERROR rows, {parse_errors} PARSE_ERROR rows — "
              f"these are excluded from K-alpha/pairwise calcs via NaN encoding but "
              f"will need re-running before this task can be finalised.")


    # ── 3b. Model x model pairwise matrix (LLM only) ───────────────────────
    print("\n" + "\u2550" * 70)
    print("3b. MODEL x MODEL PAIRWISE AGREEMENT MATRIX  (3 mid-tier raters)")
    print("\u2550" * 70)

    llm_models = sorted(df_llm["model"].unique())
    model_modes = {m: model_mode_series(df_llm, m) for m in llm_models}

    prov_col = next((c for c in df.columns if c.lower().startswith("prov")), None)

    pw_matrix = pd.DataFrame(index=llm_models, columns=llm_models, dtype=float)
    for m1, m2 in itertools.combinations(llm_models, 2):
        common = model_modes[m1].index.intersection(model_modes[m2].index)
        pa = pairwise_agreement(model_modes[m1].loc[common], model_modes[m2].loc[common])
        pw_matrix.loc[m1, m2] = pa
        pw_matrix.loc[m2, m1] = pa
    for m in llm_models:
        pw_matrix.loc[m, m] = 1.0

    print(pw_matrix.round(4).to_string())

    # ── 6. Full cross-model agreement (all 3 mid-tier raters pooled) ───────
    print("\n" + "\u2550" * 70)
    print("6.  FULL CROSS-MODEL AGREEMENT  (all 3 mid-tier raters, single run/temp)")
    print("\u2550" * 70)

    full_row = summarise_agreement(df_llm, "all_mid_tier_raters")
    full_df = pd.DataFrame([full_row])
    print(full_df.to_string(index=False))

    # ── 7. Agreement vs human / reference register (optional) ──────────────
    print("\n" + "\u2550" * 70)
    print("7.  AGREEMENT WITH HUMAN / REFERENCE ANNOTATION")
    print("    Raw % agreement + Krippendorff alpha nominal (human + model jointly)")
    print("\u2550" * 70)

    human_vs_df = pd.DataFrame()
    if _human is not None:
        human_vs_rows = []
        for model, grp in df_llm.groupby("model"):
            model_mode = (
                grp.groupby("task_id")["classification"]
                .agg(lambda x: x.mode().iloc[0])
                .reset_index()
                .rename(columns={"classification": "model_label"})
            )
            merged = model_mode.merge(
                _human.rename(columns={"classification": "human_label"}),
                on="task_id",
                how="inner",
            )
            pa = pairwise_agreement(merged["model_label"], merged["human_label"])

            long_human = pd.DataFrame({
                "task_id": merged["task_id"],
                "classification": merged["human_label"],
                "model": HUMAN_MODEL,
            })
            long_model = pd.DataFrame({
                "task_id": merged["task_id"],
                "classification": merged["model_label"],
                "model": model,
            })
            combined = pd.concat([long_human, long_model], ignore_index=True)
            p_num = pivot_numeric(combined)
            alpha_hat, ci_lo, ci_hi = krippendorff_nominal_ci(p_num.values.T)

            human_vs_rows.append({
                "model": model,
                "n_tasks": len(merged),
                "pct_agreement_with_human": round(pa, 4),
                "krippendorff_alpha_nominal": round(alpha_hat, 4) if not np.isnan(alpha_hat) else np.nan,
                "alpha_95ci_lower": round(ci_lo, 4) if not np.isnan(ci_lo) else np.nan,
                "alpha_95ci_upper": round(ci_hi, 4) if not np.isnan(ci_hi) else np.nan,
            })

        human_vs_df = pd.DataFrame(human_vs_rows).sort_values(
            "krippendorff_alpha_nominal", ascending=False
        )
        print(human_vs_df.to_string(index=False))
    else:
        print("  Skipped — no human/reference file supplied.")

    # ── 8. Per-subprocess: consensus strength ───────────────────────────────
    print("\n" + "\u2550" * 70)
    print("8.  DISAGREEMENT BREAKDOWN BY SUBPROCESS  (all raters incl. human if present)")
    print("    Primary metric: consensus_strength (mode-share, 0 -> 1)")
    print("\u2550" * 70)

    p_txt_full = pivot_text(df_all)

    task_consensus = p_txt_full.apply(consensus_strength, axis=1).rename("consensus_strength")
    task_majority = p_txt_full.apply(
        lambda row: row.dropna().mode().iloc[0] if len(row.dropna()) > 0 else np.nan,
        axis=1,
    ).rename("majority_label")
    task_n_raters = p_txt_full.apply(lambda row: row.dropna().nunique(), axis=1)

    task_summary = pd.concat([task_majority, task_consensus], axis=1).reset_index()
    task_summary["full_agreement"] = (task_n_raters == 1).values
    # Reconciliation flag mirrors the workflow-doc rule: ANY disagreement
    # on this axis sends the task to the reconciliation round.
    task_summary["needs_reconciliation"] = ~task_summary["full_agreement"]

    subprocess_agree = (
        task_summary.groupby("majority_label")
        .agg(
            n_tasks=("task_id", "count"),
            mean_consensus_strength=("consensus_strength", "mean"),
            full_agreement_rate=("full_agreement", "mean"),
        )
        .reset_index()
        .sort_values("mean_consensus_strength")
    )
    subprocess_agree["mean_consensus_strength"] = subprocess_agree["mean_consensus_strength"].round(4)
    subprocess_agree["full_agreement_rate"] = subprocess_agree["full_agreement_rate"].round(4)
    print(subprocess_agree.to_string(index=False))

    n_reconcile = int(task_summary["needs_reconciliation"].sum())
    n_total_tasks = len(task_summary)
    print(f"\n  -> {n_reconcile} / {n_total_tasks} tasks "
          f"({n_reconcile / n_total_tasks:.1%}) show disagreement on at least one "
          f"axis and enter the reconciliation round.")

    # ── 8b. Confusion / transition matrix ────────────────────────────────
    print("\n" + "\u2500" * 70)
    print("8b. SUBPROCESS CONFUSION MATRIX  (all raters incl. human if present)")
    print("""
    What it shows:
      Each cell (row=A, col=B) counts how many times a rater assigned label B
      to a task whose MAJORITY label is A. The diagonal = agreements;
      off-diagonal = disagreements. Reading across a row tells you *where*
      the minority votes went when raters diverged from the consensus on
      subprocess A. A large off-diagonal cell at (A, B) means the A<->B
      boundary is the dominant source of confusion.
    """)

    p_txt_reset = p_txt_full.copy()
    p_txt_reset["majority_label"] = task_majority.values

    confusion_pairs = []
    rater_cols = [c for c in p_txt_reset.columns if c != "majority_label"]
    for _, row in p_txt_reset.iterrows():
        majority = row["majority_label"]
        if pd.isna(majority):
            continue
        for rater in rater_cols:
            individual = row[rater]
            if not pd.isna(individual):
                confusion_pairs.append((majority, individual))

    confusion_df = pd.DataFrame(confusion_pairs, columns=["majority_label", "rater_label"])

    confusion_matrix = (
        confusion_df.groupby(["majority_label", "rater_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=SUBPROCESS_ORDER, columns=SUBPROCESS_ORDER, fill_value=0)
    )
    confusion_pct = confusion_matrix.div(confusion_matrix.sum(axis=1), axis=0).round(3)

    print(confusion_matrix.to_string())
    print("\n  Row-normalised (fraction of votes on each majority-label task):")
    print(confusion_pct.to_string())

    # ── 9. Punt rate per rater ───────────────────────────────────────────
    print("\n" + "\u2550" * 70)
    print("9.  PUNT RATE BY RATER  (fraction labelled AMBIGUOUS or OUT_OF_SCOPE)")
    print("\u2550" * 70)

    df_all_tmp = df_all.copy()
    df_all_tmp["_rater"] = _build_rater_id(df_all_tmp)
    df_all_tmp["is_punt"] = df_all_tmp["classification"].isin(PUNT_LABELS)

    punt_rows = []
    for rater_id, grp in df_all_tmp.groupby("_rater"):
        model = grp["model"].iloc[0]
        n_total = len(grp)
        n_punt = grp["is_punt"].sum()
        n_ambiguous = (grp["classification"] == "AMBIGUOUS").sum()
        n_oos = (grp["classification"] == "OUT_OF_SCOPE").sum()
        punt_rows.append({
            "rater": rater_id,
            "model": model,
            "n_tasks": n_total,
            "n_punt": n_punt,
            "punt_rate": round(n_punt / n_total, 4) if n_total else np.nan,
            "ambiguous_rate": round(n_ambiguous / n_total, 4) if n_total else np.nan,
            "oos_rate": round(n_oos / n_total, 4) if n_total else np.nan,
        })

    punt_df = pd.DataFrame(punt_rows).sort_values(["model", "punt_rate"])
    print(punt_df.to_string(index=False))

    # ── 10. NEW — Task-volume distribution per subprocess + Pearson r ──────
    print("\n" + "\u2550" * 70)
    print("10. TASK VOLUME PER SUBPROCESS, BY RATER  +  PEARSON CORRELATION")
    print("    (workflow-doc check: 'Pearson correlation total task volumes")
    print("     per subprocess')")
    print("\u2550" * 70)

    df_vol = df_all.copy()
    df_vol["_rater"] = _build_rater_id(df_vol)

    volume_table = (
        df_vol.groupby(["_rater", "classification"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=SUBPROCESS_ORDER, fill_value=0)
    )
    print("\n  Raw counts per subprocess (rows = raters):")
    print(volume_table.to_string())

    # Pearson correlation between raters' subprocess-volume profiles —
    # each rater is a vector of counts across the 8 labels; this measures
    # whether raters agree on the *shape* of the distribution (which
    # subprocesses are dense/sparse) even before checking task-level
    # agreement.
    volume_corr = volume_table.T.corr(method="pearson").round(4)
    print("\n  Pearson correlation matrix (rater x rater, over subprocess volumes):")
    print(volume_corr.to_string())

    # Also report each LLM rater's volume profile against the human/
    # reference register, if supplied — directly answers "do model
    # subprocess volumes track the validated register's volumes?"
    volume_vs_human = pd.DataFrame()
    if _human is not None:
        human_counts = (
            _human["classification"].value_counts()
            .reindex(SUBPROCESS_ORDER, fill_value=0)
        )
        rows = []
        for rater in volume_table.index:
            if rater == HUMAN_MODEL:
                continue
            r = np.corrcoef(volume_table.loc[rater].values, human_counts.values)[0, 1]
            rows.append({"rater": rater, "pearson_r_vs_human_volume": round(r, 4)})
        volume_vs_human = pd.DataFrame(rows).sort_values(
            "pearson_r_vs_human_volume", ascending=False
        )
        print("\n  Pearson r of each rater's subprocess-volume profile vs. human/reference:")
        print(volume_vs_human.to_string(index=False))

    # ── Save all outputs ──────────────────────────────────────────────────
    out_path = Path("production_classification_text_labelled.csv")
    df.to_csv(out_path, index=False)
    print(f"\n\u2713  Text-labelled data saved \u2192 {out_path}")

    summary_path = Path("production_agreement_summary.xlsx")
    with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
        pw_matrix.to_excel(writer, sheet_name="3b_model_pairwise_matrix")
        full_df.to_excel(writer, sheet_name="6_all_mid_tier_raters", index=False)
        if not human_vs_df.empty:
            human_vs_df.to_excel(writer, sheet_name="7_vs_human", index=False)
        subprocess_agree.to_excel(writer, sheet_name="8_by_subprocess", index=False)
        task_summary.to_excel(writer, sheet_name="task_level_summary", index=False)
        confusion_matrix.to_excel(writer, sheet_name="8b_confusion_counts")
        confusion_pct.to_excel(writer, sheet_name="8b_confusion_rowpct")
        punt_df.to_excel(writer, sheet_name="9_punt_rates", index=False)
        volume_table.to_excel(writer, sheet_name="10_subprocess_volumes")
        volume_corr.to_excel(writer, sheet_name="10_pearson_corr_matrix")
        if not volume_vs_human.empty:
            volume_vs_human.to_excel(writer, sheet_name="10_pearson_vs_human", index=False)

    print(f"\u2713  Summary tables saved   \u2192 {summary_path}\n")


# ── Entry point ───────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(description="R2R Phase 1 Production-Run Agreement Analysis")
    parser.add_argument(
        "--input", required=True,
        help="Path to the unified long-format CSV (task_id, task_statement, tier, "
             "provider, model, temperature, run_id, classification, reasoning, ...). "
             "Human rows should already be present with model='human_domain_expert' "
             "(this is the normal case — same file shape as unified_rater_long_format.csv).",
    )
    parser.add_argument(
        "--human", default=None,
        help="Only needed if human rows are NOT already in --input. Path to a "
             "separate CSV with columns task_id, classification (pre-mapped to the "
             "production label scheme); these rows get tagged "
             "model='human_domain_expert' and merged in before analysis.",
    )
    args = parser.parse_args()

    print(f"\nLoading: {args.input}")
    df = pd.read_csv(args.input)
    print(f"  Rows: {len(df):,}  |  Columns: {list(df.columns)}")

    if args.human:
        print(f"Loading separate human/reference file: {args.human}")
        human_df = pd.read_csv(args.human)
        missing = [c for c in ["task_id", "classification"] if c not in human_df.columns]
        if missing:
            raise ValueError(f"--human file missing required columns: {missing}")
        human_df = human_df[["task_id", "classification"]].copy()
        human_df["model"] = HUMAN_MODEL
        human_df["provider"] = "human"
        human_df["tier"] = "human_expert"
        human_df["temperature"] = np.nan
        human_df["run_id"] = np.nan
        df = pd.concat([df, human_df], ignore_index=True)
        print(f"  Merged {len(human_df):,} human/reference rows.")

    run_analysis(df)


df = pd.read_csv("june 23 unified_rater_long_format.csv")
run_analysis(df)

  ✓  Human/reference annotations found: 212 tasks

══════════════════════════════════════════════════════════════════════
3b. MODEL x MODEL PAIRWISE AGREEMENT MATRIX  (3 mid-tier raters)
══════════════════════════════════════════════════════════════════════
                       claude-sonnet-4-6  gemini-3.1-flash-lite  gpt-5.4-mini
claude-sonnet-4-6                 1.0000                 0.8774        0.8066
gemini-3.1-flash-lite             0.8774                 1.0000        0.8208
gpt-5.4-mini                      0.8066                 0.8208        1.0000

══════════════════════════════════════════════════════════════════════
6.  FULL CROSS-MODEL AGREEMENT  (all 3 mid-tier raters, single run/temp)
══════════════════════════════════════════════════════════════════════
              group  n_raters  n_tasks  krippendorff_alpha_nominal  alpha_95ci_lower  alpha_95ci_upper  mean_pairwise_agreement  full_agreement_rate
all_mid_tier_raters         3      212                      0.738